In [1]:
import numpy as np
import pandas as pd
import copy
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams.update({'figure.dpi':120,'font.size':10,
                     'axes.spines.top':False,'axes.spines.right':False})
print(f'Device: {DEVICE}')
print('설정 완료')

c:\Users\kevin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
설정 완료


In [2]:
BASE = r'c:\Users\kevin\OneDrive\Desktop\AISO\Elliptic Bitcoin\elliptic_bitcoin_dataset'

print('로딩 중...')
feat_df = pd.read_csv(f'{BASE}/elliptic_txs_features.csv', header=None)
cls_df  = pd.read_csv(f'{BASE}/elliptic_txs_classes.csv')
edge_df = pd.read_csv(f'{BASE}/elliptic_txs_edgelist.csv')

feat_df.columns = ['txId'] + [f'f{i}' for i in range(1, 167)]
feat_df['timestep'] = feat_df['f1'].astype(int)

df = feat_df.merge(cls_df, on='txId')
print(f'전체: {len(df):,}노드 | 피처: 166 | 엣지: {len(edge_df):,}')
print('\nclass 분포:')
print(df['class'].value_counts().to_string())
print(f'\n타임스텝: {df["timestep"].min()} ~ {df["timestep"].max()}')

로딩 중...
전체: 203,769노드 | 피처: 166 | 엣지: 234,355

class 분포:
class
unknown    157205
2           42019
1            4545

타임스텝: 1 ~ 49


In [3]:
# ── labeled only + temporal split ────────────────────────────
labeled = df[df['class'] != 'unknown'].copy().reset_index(drop=True)
labeled['y'] = (labeled['class'] == '1').astype(int)

feat_cols = [f'f{i}' for i in range(1, 167)]
X_all = labeled[feat_cols].values.astype(float)
y_all = labeled['y'].values
ts_all = labeled['timestep'].values
N_NODES = len(labeled)

print(f'Labeled: {N_NODES:,}  (illicit={y_all.sum():,}, licit={(y_all==0).sum():,})')

# node id → local index
txid_to_idx = {txid: i for i, txid in enumerate(labeled['txId'].values)}

# edge index (labeled 노드 간 엣지만)
s_arr = edge_df.iloc[:,0].values
d_arr = edge_df.iloc[:,1].values
s_map = np.array([txid_to_idx.get(t, -1) for t in s_arr])
d_map = np.array([txid_to_idx.get(t, -1) for t in d_arr])
valid = (s_map >= 0) & (d_map >= 0)
s_v, d_v = s_map[valid], d_map[valid]
edge_index = torch.tensor(
    [np.concatenate([s_v, d_v]), np.concatenate([d_v, s_v])],
    dtype=torch.long
)
print(f'유효 엣지: {valid.sum():,}개 (양방향 {edge_index.shape[1]:,})')

# temporal masks
train_mask_all = ts_all <= 34
test_mask      = ts_all > 34

# training pool
train_norm_idx = np.where(train_mask_all & (y_all==0))[0]
train_anom_idx = np.where(train_mask_all & (y_all==1))[0]

N_NORMAL = min(10000, len(train_norm_idx))
N_SEEN   = min(1000,  len(train_anom_idx))
rng = np.random.RandomState(SEED)
sel_n = rng.choice(train_norm_idx, N_NORMAL, replace=False)

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# PCA for optimization samplers
pca = PCA(n_components=30, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
X_anom_pca = X_pca[train_anom_idx]

print(f'\n학습 licit pool: {N_NORMAL:,} | illicit pool: {len(train_anom_idx):,} → 선택: {N_SEEN:,}')
print(f'테스트: {test_mask.sum():,}  illicit={y_all[test_mask].sum():,}')

Labeled: 46,564  (illicit=4,545, licit=42,019)
유효 엣지: 36,624개 (양방향 73,248)

학습 licit pool: 10,000 | illicit pool: 3,462 → 선택: 1,000
테스트: 16,670  illicit=1,083


In [4]:

# ── GCN 모델 + 평가 함수 (Subgraph 방식) ─────────────────────
# Yelp 재현: sampler가 선택한 노드로 induced subgraph 구성
# → sampler마다 다른 그래프 토폴로지 → GCN이 다른 구조를 학습
results = {}
preds   = {}

ei_np = edge_index.numpy()   # shape (2, E), global indices

# 빠른 local index 변환용 lookup 배열
_LOOKUP = np.full(N_NODES, -1, dtype=np.int32)

class GCN(torch.nn.Module):
    def __init__(self, in_ch, hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin   = torch.nn.Linear(hidden, 2)
        self.drop  = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.drop, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.drop, training=self.training)
        return self.lin(x)

def evaluate_gnn(sel_pool_idx, label=''):
    """
    sel_pool_idx: indices into train_anom_idx pool
    Subgraph = selected illicit (t≤34) + sel_n licit (t≤34) + ALL test nodes (t>34)
    Induced edges between subgraph nodes only
    → sampler마다 다른 그래프 토폴로지 (Yelp 구조와 동일)
    """
    uniq_anom = np.unique(sel_pool_idx)
    sel_anom_global = train_anom_idx[uniq_anom]
    test_global     = np.where(test_mask)[0]

    # subgraph node 집합
    sub_nodes = np.unique(np.concatenate([sel_n, sel_anom_global, test_global]))
    n_sub = len(sub_nodes)

    # global → subgraph local index 변환
    _LOOKUP[:] = -1
    _LOOKUP[sub_nodes] = np.arange(n_sub)
    src_loc = _LOOKUP[ei_np[0]]
    dst_loc = _LOOKUP[ei_np[1]]
    valid   = (src_loc >= 0) & (dst_loc >= 0)
    sub_ei  = torch.tensor([src_loc[valid], dst_loc[valid]], dtype=torch.long).to(DEVICE)

    # features / labels in subgraph order
    sub_X = torch.from_numpy(X_scaled[sub_nodes]).float().to(DEVICE)
    sub_y = torch.from_numpy(y_all[sub_nodes]).long().to(DEVICE)

    # train / test masks (subgraph-local)
    train_set = set(np.concatenate([sel_n, sel_anom_global]).tolist())
    test_set  = set(test_global.tolist())
    tr_mask = torch.tensor([g in train_set for g in sub_nodes], dtype=torch.bool).to(DEVICE)
    te_mask = torch.tensor([g in test_set  for g in sub_nodes], dtype=torch.bool).to(DEVICE)

    n0 = int((y_all[sub_nodes][tr_mask.cpu().numpy()]==0).sum())
    n1 = int((y_all[sub_nodes][tr_mask.cpu().numpy()]==1).sum())
    cw = torch.tensor([1.0, n0/max(n1,1)], dtype=torch.float).to(DEVICE)

    torch.manual_seed(SEED)
    model = GCN(X_scaled.shape[1]).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    best_loss, best_state, patience = float('inf'), None, 0
    for ep in range(200):
        model.train(); opt.zero_grad()
        out  = model(sub_X, sub_ei)
        loss = F.cross_entropy(out[tr_mask], sub_y[tr_mask], weight=cw)
        loss.backward(); opt.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
        if patience >= 20: break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        prob = F.softmax(model(sub_X, sub_ei), dim=1)[:,1].cpu().numpy()
    pred = (prob >= 0.5).astype(int)

    te_cpu = te_mask.cpu().numpy()
    res = {
        'PR-AUC': average_precision_score(y_all[sub_nodes][te_cpu], prob[te_cpu]),
        'F1':     f1_score(y_all[sub_nodes][te_cpu], pred[te_cpu], zero_division=0),
        'AUC':    roc_auc_score(y_all[sub_nodes][te_cpu], prob[te_cpu]),
        'n_edges': int(valid.sum()),
    }
    if label:
        preds[label.strip()] = (sub_nodes[te_cpu], prob[te_cpu])
        print(f'  {label:<22} PR-AUC={res["PR-AUC"]:.4f}  F1={res["F1"]:.4f}'
              f'  AUC={res["AUC"]:.4f}  edges={res["n_edges"]:,}')
    return res

print('GCN + Subgraph 평가 함수 준비 완료')
print('(sampler마다 다른 induced subgraph → 다른 그래프 토폴로지)')


GCN + Subgraph 평가 함수 준비 완료
(sampler마다 다른 induced subgraph → 다른 그래프 토폴로지)


In [5]:
# ── 샘플러 정의 ──────────────────────────────────────────────
N_AG = 20; N_IT = 80; ALPHA = 0.2
N_TYPES = 8; BETA = 0.08; W_REPEL = 2.0; M_LOW = -0.5

def _norm(X):
    mn,mx = X.min(0),X.max(0)
    return (X-mn)/np.where(mx-mn>1e-8,mx-mn,1.0)

def run_random(X_a, n, seed):
    return np.random.RandomState(seed).choice(len(X_a), n, replace=True)

def run_kmeans(X_a, n, seed, k=8):
    km = KMeans(k, random_state=seed, n_init=5).fit(X_a)
    rng2 = np.random.RandomState(seed); idx = []
    for c in range(k):
        pool = np.where(km.labels_==c)[0]
        if len(pool): idx.extend(rng2.choice(pool, n//k, replace=True))
    while len(idx)<n: idx.append(rng2.randint(len(X_a)))
    return np.array(idx[:n])

def run_topdensity(X_a, n, seed, k=5):
    nn = NearestNeighbors(n_neighbors=min(k+1,len(X_a))).fit(X_a)
    d,_ = nn.kneighbors(X_a)
    density = 1.0/(d[:,1:].mean(1)+1e-8)
    probs = density/density.sum()
    return np.random.RandomState(seed).choice(len(X_a), n, replace=True, p=probs)

def run_pso(X_a, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_a); N_a,D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = np.array([-np.min(np.linalg.norm(Xn-X[i],axis=1)) for i in range(N_AG)])
    gi = np.argmax(pS); gX = X[gi].copy(); visit = np.zeros(N_a)
    for _ in range(N_IT):
        r1,r2 = rng2.rand(N_AG,D),rng2.rand(N_AG,D)
        V = 0.729*V+1.494*r1*(pX-X)+1.494*r2*(gX-X)
        X = np.clip(X+ALPHA*V,0,1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc = -np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

def run_aiso(X_a, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_a); N_a,D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(N_TYPES),N_AG)
    M = rng2.uniform(M_LOW,W_REPEL,(N_TYPES,N_TYPES))
    visit = np.zeros(N_a); w_r = W_REPEL
    for t in range(N_IT):
        if t%10==0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r = 1.0+3.0*np.exp(-div/0.12)
        C = W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci = C[i].copy(); ci[i]=0
            ta = np.argsort(ci)[-3:]; tr2 = np.argsort(ci)[:3]
            Fv = sum(ci[j]*(X[j]-X[i]) for j in ta)\
               + w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2); Fv/=6.0
            nn = np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja = ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

print('샘플러 준비 완료')

샘플러 준비 완료


In [6]:

# ── 7개 메서드 실행 ───────────────────────────────────────────
def run(name, idx):
    results[name] = evaluate_gnn(idx, name)

print('='*65)
print('카테고리 1: 베이스라인')
print('  원본(불균형)...', end=' ', flush=True)
run('원본(불균형)', np.random.RandomState(SEED).choice(len(train_anom_idx), N_SEEN, replace=False))

print('='*65)
print('카테고리 2: 룰 기반')
print('  Random...', end=' ', flush=True)
run('Random',      run_random(X_anom_pca, N_SEEN, SEED))
print('  K-Means...', end=' ', flush=True)
run('K-Means',     run_kmeans(X_anom_pca, N_SEEN, SEED))
print('  Top-density...', end=' ', flush=True)
run('Top-density', run_topdensity(X_anom_pca, N_SEEN, SEED))

print('='*65)
print('카테고리 3: 최적화 샘플링 (PCA-30D)')
print('  PSO...', end=' ', flush=True)
run('PSO',  run_pso(X_anom_pca,  N_SEEN, SEED))
print('  AISO...', end=' ', flush=True)
run('AISO', run_aiso(X_anom_pca, N_SEEN, SEED))

print('='*65)
print(f'완료! 총 {len(results)}개 메서드')


카테고리 1: 베이스라인
  원본(불균형)...   원본(불균형)                PR-AUC=0.5752  F1=0.5410  AUC=0.8668  edges=33,596
카테고리 2: 룰 기반
  Random...   Random                 PR-AUC=0.5538  F1=0.5596  AUC=0.8523  edges=33,548
  K-Means...   K-Means                PR-AUC=0.3587  F1=0.4370  AUC=0.8191  edges=33,648
  Top-density...   Top-density            PR-AUC=0.5701  F1=0.5204  AUC=0.8336  edges=33,304
카테고리 3: 최적화 샘플링 (PCA-30D)
  PSO...   PSO                    PR-AUC=0.6030  F1=0.6058  AUC=0.8631  edges=33,518
  AISO...   AISO                   PR-AUC=0.6002  F1=0.5775  AUC=0.8589  edges=33,438
완료! 총 6개 메서드


In [7]:
# ── 결과 시각화 ───────────────────────────────────────────────
CATEGORIES = {
    '베이스라인'   : ['원본(불균형)', 'Class Weight'],
    '룰 기반'      : ['Random', 'K-Means', 'Top-density'],
    '최적화 샘플링': ['PSO', 'AISO'],
}
CAT_C = {
    '베이스라인'   : '#888888',
    '룰 기반'      : '#4C72B0',
    '최적화 샘플링': '#C44E52',
}
MC = {m:CAT_C[c] for c,ms in CATEGORIES.items() for m in ms}
MC['AISO'] = '#8B0000'

ranking = sorted(results, key=lambda k: results[k]['PR-AUC'], reverse=True)

fig, axes = plt.subplots(1,3,figsize=(22,6))
for ax, metric in zip(axes[:2], ['PR-AUC','F1']):
    vals   = [results[m][metric] for m in ranking]
    colors = [MC.get(m,'#aaa') for m in ranking]
    bars   = ax.barh(ranking[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
    for bar,v in zip(bars,vals[::-1]):
        ax.text(v+0.002,bar.get_y()+bar.get_height()/2,
                f'{v:.4f}',va='center',fontsize=8.5)
    ax.set_xlabel(metric)
    ax.set_title(f'Elliptic Bitcoin GNN — {metric}\n'
                 f'(GCN | 실제 엣지 구조 | 시간 기반 train/test 분리)')
    ax.set_xlim(0, max(vals)*1.18)

ax = axes[2]
for m in ranking:
    ax.scatter(results[m]['AUC'],results[m]['PR-AUC'],
               s=160,color=MC.get(m,'#aaa'),zorder=5)
    ax.annotate(m,(results[m]['AUC'],results[m]['PR-AUC']),
                xytext=(4,4),textcoords='offset points',fontsize=9)
ax.set_xlabel('AUC-ROC'); ax.set_ylabel('PR-AUC')
ax.set_title('AUC vs PR-AUC')

from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(facecolor=CAT_C[c],label=c) for c in CAT_C],
               fontsize=9, loc='lower right')
plt.suptitle('Elliptic Bitcoin GNN Showdown\n'
             '(GCN + 실제 그래프 엣지 | sampling_showdown 구조 재현)',
             fontweight='bold',fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_gnn_showdown_main.png',bbox_inches='tight',dpi=120)
plt.show()

print('\n'+'='*65)
print(f'  {"전략":<18} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}  카테고리')
print('-'*65)
for rank,m in enumerate(ranking,1):
    cat = next((c for c,ms in CATEGORIES.items() if m in ms),'?')
    star = ' ★' if m=='AISO' else ''
    print(f'  {rank:>2}위 {m:<16} {results[m]["PR-AUC"]:>8.4f}'
          f' {results[m]["F1"]:>8.4f} {results[m]["AUC"]:>8.4f}  {cat}{star}')
print('='*65)


  전략                   PR-AUC       F1      AUC  카테고리
-----------------------------------------------------------------
   1위 PSO                0.6030   0.6058   0.8631  최적화 샘플링
   2위 AISO               0.6002   0.5775   0.8589  최적화 샘플링 ★
   3위 원본(불균형)            0.5752   0.5410   0.8668  베이스라인
   4위 Top-density        0.5701   0.5204   0.8336  룰 기반
   5위 Random             0.5538   0.5596   0.8523  룰 기반
   6위 K-Means            0.3587   0.4370   0.8191  룰 기반


In [8]:
# ── Mode Collapse: PSO vs AISO (샘플링 단계) ─────────────────
TRACK_EVERY = 8
track_iters = list(range(0, N_IT+1, TRACK_EVERY))

def entropy(counts):
    p = counts/(counts.sum()+1e-9); p=p[p>0]
    return -np.sum(p*np.log(p+1e-9))

def track_pso(Xn, seed):
    rng2=np.random.RandomState(seed)
    N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V=np.zeros_like(X); pX=X.copy()
    pS=-np.ones(N_AG)*1e9; gi=0; gX=X[0].copy(); visit=np.zeros(N_a)
    disp_hist=[]; vent_hist=[]
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d=np.mean([np.linalg.norm(X[i]-X[j])
                       for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
        r1,r2=rng2.rand(N_AG,D),rng2.rand(N_AG,D)
        V=0.729*V+1.494*r1*(pX-X)+1.494*r2*(gX-X)
        X=np.clip(X+ALPHA*V,0,1)
        for i in range(N_AG):
            nn=np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc=-np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j])
                               for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    return disp_hist, vent_hist

def track_aiso(Xn, seed):
    rng2=np.random.RandomState(seed)
    N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W=rng2.dirichlet(np.ones(N_TYPES),N_AG)
    M=rng2.uniform(M_LOW,W_REPEL,(N_TYPES,N_TYPES))
    visit=np.zeros(N_a); w_r=W_REPEL
    disp_hist=[]; vent_hist=[]; went_hist=[]
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d=np.mean([np.linalg.norm(X[i]-X[j])
                       for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
            went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9)) for i in range(N_AG)]))
        if t%10==0:
            dv=np.mean([np.linalg.norm(X[i]-X[j])
                        for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r=1.0+3.0*np.exp(-dv/0.12)
        C=W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci=C[i].copy(); ci[i]=0
            ta=np.argsort(ci)[-3:]; tr2=np.argsort(ci)[:3]
            Fv=sum(ci[j]*(X[j]-X[i]) for j in ta)\
              +w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2); Fv/=6.0
            nn=np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja=ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j])
                               for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9)) for i in range(N_AG)]))
    return disp_hist, vent_hist, went_hist

print('Mode Collapse 추적 중...')
Xn_t = _norm(X_anom_pca)
print('  PSO...', end=' ', flush=True)
pso_disp,pso_vent = track_pso(Xn_t, SEED); print('완료')
print('  AISO...', end=' ', flush=True)
aiso_disp,aiso_vent,aiso_went = track_aiso(Xn_t, SEED); print('완료')

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].plot(track_iters,pso_disp,'b-o',ms=4,label='PSO',lw=2)
axes[0].plot(track_iters,aiso_disp,'r-o',ms=4,label='AISO',lw=2)
axes[0].set_title('Agent Dispersion',fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(track_iters,pso_vent,'b-o',ms=4,label='PSO',lw=2)
axes[1].plot(track_iters,aiso_vent,'r-o',ms=4,label='AISO',lw=2)
axes[1].set_title('Visit Count Entropy H(visit)',fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(track_iters,aiso_went,'r-o',ms=4,label='AISO W-entropy',lw=2)
axes[2].axhline(np.log(N_TYPES),color='gray',ls='--',label=f'Max H=log({N_TYPES})')
axes[2].set_title('AISO Type Entropy H(W)',fontweight='bold')
axes[2].set_xlabel('Iteration'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Mode Collapse 분석: PSO vs AISO (Elliptic GNN 샘플링 단계)',
             fontweight='bold',fontsize=11)
plt.tight_layout()
plt.savefig('elliptic_gnn_mode_collapse.png',bbox_inches='tight',dpi=120)
plt.show()

print(f'\n최종 dispersion : PSO={pso_disp[-1]:.4f}  AISO={aiso_disp[-1]:.4f}')
print(f'최종 visit H    : PSO={pso_vent[-1]:.4f}  AISO={aiso_vent[-1]:.4f}')
print(f'최종 W entropy  : AISO={aiso_went[-1]:.4f} (max={np.log(N_TYPES):.4f})')

Mode Collapse 추적 중...
  PSO... 완료
  AISO... 완료

최종 dispersion : PSO=0.3478  AISO=0.4950
최종 visit H    : PSO=3.0605  AISO=3.2139
최종 W entropy  : AISO=1.8474 (max=2.0794)


In [9]:
# ── N_TYPES 차원 탐색: AISO 최적 타입 수 찾기 (GNN) ─────────────────────────
# 주의: GCN 학습 포함 → 탐색당 ~10-30초 소요
TEST_TYPES = [4, 6, 8, 9, 12, 14, 17, 20, 24]

def run_aiso_nt(X_a, n, seed, n_types):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_a); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a, N_AG, replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(n_types), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (n_types, n_types))
    visit = np.zeros(N_a); w_r = W_REPEL
    for t in range(N_IT):
        if t % 10 == 0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1, N_AG)])
            w_r = 1.0 + 3.0 * np.exp(-div / 0.12)
        C = W @ M @ W.T; np.fill_diagonal(C, 0)
        for i in range(N_AG):
            ci = C[i].copy(); ci[i] = 0
            ta = np.argsort(ci)[-3:]; tr2 = np.argsort(ci)[:3]
            Fv  = sum(ci[j] * (X[j]-X[i]) for j in ta)
            Fv += w_r * sum(ci[j] * (X[j]-X[i]) for j in tr2)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn - np.clip(X[i]+ALPHA*Fv, 0, 1), axis=1))
            X[i] = Xn[nn]; visit[nn] += 1.0
            bja = ta[np.argmax(ci[ta])]
            W[i] = (1-BETA)*W[i] + BETA*W[bja]; W[i] /= W[i].sum()
    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, n, replace=True, p=probs)

print('N_TYPES 차원 탐색 (Elliptic GNN)')
print(f'  현재 설정: N_TYPES={N_TYPES}  (GCN 학습 포함, 시간 소요)')
print('-'*50)
type_results_nt = {}
for nt in TEST_TYPES:
    idx = run_aiso_nt(X_anom_pca, N_SEEN, SEED, nt)
    res = evaluate_gnn(idx)
    type_results_nt[nt] = res['PR-AUC']
    marker = ' <-- 현재' if nt == N_TYPES else ''
    print(f'  N_TYPES={nt:>2}  PR-AUC={res["PR-AUC"]:.4f}  F1={res["F1"]:.4f}{marker}')

best_nt = max(type_results_nt, key=type_results_nt.get)
print('-'*50)
print(f'  최적 N_TYPES = {best_nt}  (PR-AUC={type_results_nt[best_nt]:.4f})')
print(f'  기본값({N_TYPES}) PR-AUC  = {type_results_nt[N_TYPES]:.4f}')
print(f'  개선 여지        = {type_results_nt[best_nt]-type_results_nt[N_TYPES]:+.4f}')
print(f'\n참고: PSO 결과 = {results["PSO"]["PR-AUC"]:.4f}')


N_TYPES 차원 탐색 (Elliptic GNN)
  현재 설정: N_TYPES=8  (GCN 학습 포함, 시간 소요)
--------------------------------------------------
  N_TYPES= 4  PR-AUC=0.5127  F1=0.4954
  N_TYPES= 6  PR-AUC=0.5673  F1=0.5521
  N_TYPES= 8  PR-AUC=0.6002  F1=0.5775 <-- 현재
  N_TYPES= 9  PR-AUC=0.5717  F1=0.5467
  N_TYPES=12  PR-AUC=0.3684  F1=0.4628
  N_TYPES=14  PR-AUC=0.6378  F1=0.6068
  N_TYPES=17  PR-AUC=0.5757  F1=0.5865
  N_TYPES=20  PR-AUC=0.3933  F1=0.4710
  N_TYPES=24  PR-AUC=0.4946  F1=0.5151
--------------------------------------------------
  최적 N_TYPES = 14  (PR-AUC=0.6378)
  기본값(8) PR-AUC  = 0.6002
  개선 여지        = +0.0377

참고: PSO 결과 = 0.6030


In [10]:
# ── N_TYPES=14 최적값으로 AISO 재평가 + 전체 랭킹 업데이트 ──────────────────
BEST_N_TYPES = best_nt  # 차원 탐색에서 찾은 최적값 (14)
print(f'최적 N_TYPES = {BEST_N_TYPES} 로 AISO 재평가')
print('='*65)

# 기존 N_TYPES 백업 후 최적값으로 교체
_orig_N_TYPES = N_TYPES
N_TYPES = BEST_N_TYPES

idx_best = run_aiso(X_anom_pca, N_SEEN, SEED)
res_best = evaluate_gnn(idx_best, f'AISO(N_TYPES={BEST_N_TYPES})')
N_TYPES = _orig_N_TYPES  # 복원

# 전체 결과에 추가
results_tuned = dict(results)
results_tuned[f'AISO(N_TYPES={BEST_N_TYPES})'] = res_best

ranking_tuned = sorted(results_tuned, key=lambda k: results_tuned[k]['PR-AUC'], reverse=True)

CATEGORIES_T = {
    '베이스라인'   : ['원본(불균형)'],
    '룰 기반'      : ['Random', 'K-Means', 'Top-density'],
    '최적화 샘플링': ['PSO', 'AISO', f'AISO(N_TYPES={BEST_N_TYPES})'],
}
CAT_C_T = {'베이스라인':'#888888','룰 기반':'#4C72B0','최적화 샘플링':'#C44E52'}
MC_T = {m:CAT_C_T[c] for c,ms in CATEGORIES_T.items() for m in ms}
MC_T['AISO'] = '#C44E52'
MC_T[f'AISO(N_TYPES={BEST_N_TYPES})'] = '#8B0000'

fig, axes = plt.subplots(1,2,figsize=(16,6))
for ax, metric in zip(axes, ['PR-AUC','F1']):
    vals   = [results_tuned[m][metric] for m in ranking_tuned]
    colors = [MC_T.get(m,'#aaa') for m in ranking_tuned]
    bars   = ax.barh(ranking_tuned[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
    for bar,v in zip(bars,vals[::-1]):
        ax.text(v+0.002,bar.get_y()+bar.get_height()/2,f'{v:.4f}',va='center',fontsize=8.5)
    ax.set_xlabel(metric)
    ax.set_title(f'Elliptic GNN — {metric}\n(N_TYPES 튜닝 전/후 비교)')
    ax.set_xlim(0, max(vals)*1.2)
    ax.axvline(results['PSO']['PR-AUC'], color='blue', ls='--', alpha=0.4, label='PSO baseline')
    ax.legend(fontsize=8)

plt.suptitle(f'N_TYPES 튜닝 효과: AISO(8) vs AISO({BEST_N_TYPES}) vs PSO',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_gnn_ntype_tuned.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n' + '='*65)
print(f'  {"전략":<26} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}')
print('-'*65)
for rank, m in enumerate(ranking_tuned, 1):
    star = ' ★' if 'AISO' in m else ''
    print(f'  {rank:>2}위 {m:<24} {results_tuned[m]["PR-AUC"]:>8.4f}'
          f' {results_tuned[m]["F1"]:>8.4f} {results_tuned[m]["AUC"]:>8.4f}{star}')
print('='*65)
print(f'\nPSO           : {results["PSO"]["PR-AUC"]:.4f}')
print(f'AISO (N=8)    : {results["AISO"]["PR-AUC"]:.4f}  (기존)')
print(f'AISO (N={BEST_N_TYPES})   : {res_best["PR-AUC"]:.4f}  (+{res_best["PR-AUC"]-results["PSO"]["PR-AUC"]:+.4f} vs PSO)')


최적 N_TYPES = 14 로 AISO 재평가
  AISO(N_TYPES=14)       PR-AUC=0.6378  F1=0.6068  AUC=0.8758  edges=33,446

  전략                           PR-AUC       F1      AUC
-----------------------------------------------------------------
   1위 AISO(N_TYPES=14)           0.6378   0.6068   0.8758 ★
   2위 PSO                        0.6030   0.6058   0.8631
   3위 AISO                       0.6002   0.5775   0.8589 ★
   4위 원본(불균형)                    0.5752   0.5410   0.8668
   5위 Top-density                0.5701   0.5204   0.8336
   6위 Random                     0.5538   0.5596   0.8523
   7위 K-Means                    0.3587   0.4370   0.8191

PSO           : 0.6030
AISO (N=8)    : 0.6002  (기존)
AISO (N=14)   : 0.6378  (++0.0348 vs PSO)


In [11]:
# ── 다중 시드 N_TYPES 검증: 14가 SEED=42 lucky peak인지 확인 ──────────────────
# 5개 시드 × 5개 N_TYPES → 25회 GCN 학습 (~15분 소요)
SEEDS_VAL   = [0, 7, 42, 77, 123]
TYPES_VAL   = [8, 12, 14, 17, 20]   # 8=기본, 12/20=급락, 14=최적, 17=Yelp최적

import time
seed_type_results = {nt: {} for nt in TYPES_VAL}

print('다중 시드 N_TYPES 검증 (Elliptic GNN)')
print(f'  시드: {SEEDS_VAL}')
print(f'  N_TYPES: {TYPES_VAL}')
print('=' * 55)

for nt in TYPES_VAL:
    row = []
    for sd in SEEDS_VAL:
        idx = run_aiso_nt(X_anom_pca, N_SEEN, sd, nt)
        res = evaluate_gnn(idx)
        seed_type_results[nt][sd] = res['PR-AUC']
        row.append(res['PR-AUC'])
    mean_v = np.mean(row); std_v = np.std(row)
    marker = ' <-- 현재최적' if nt == 14 else (' <-- 기본' if nt == 8 else '')
    print(f'  N_TYPES={nt:>2}  mean={mean_v:.4f} ± {std_v:.4f}  {row}{marker}')

print('=' * 55)

# 시드별 순위 확인
print('\n시드별 best N_TYPES:')
for sd in SEEDS_VAL:
    best = max(TYPES_VAL, key=lambda nt: seed_type_results[nt][sd])
    vals = {nt: f'{seed_type_results[nt][sd]:.4f}' for nt in TYPES_VAL}
    print(f'  SEED={sd:>3}  best={best}  | {vals}')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for sd in SEEDS_VAL:
    vals = [seed_type_results[nt][sd] for nt in TYPES_VAL]
    axes[0].plot(TYPES_VAL, vals, 'o-', ms=5, label=f'seed={sd}', alpha=0.7)
axes[0].axvline(14, color='red', ls='--', alpha=0.5, label='N=14 (SEED=42 최적)')
axes[0].axvline(8,  color='blue', ls='--', alpha=0.3, label='N=8 (기본)')
axes[0].set_xlabel('N_TYPES'); axes[0].set_ylabel('PR-AUC')
axes[0].set_title('N_TYPES별 PR-AUC (5개 시드)', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

means = [np.mean([seed_type_results[nt][sd] for sd in SEEDS_VAL]) for nt in TYPES_VAL]
stds  = [np.std( [seed_type_results[nt][sd] for sd in SEEDS_VAL]) for nt in TYPES_VAL]
axes[1].bar(range(len(TYPES_VAL)), means, yerr=stds, capsize=5,
            color=['#C44E52' if nt==14 else '#4C72B0' if nt==8 else '#aaa' for nt in TYPES_VAL],
            alpha=0.8)
axes[1].set_xticks(range(len(TYPES_VAL)))
axes[1].set_xticklabels([f'N={nt}' for nt in TYPES_VAL])
axes[1].set_ylabel('PR-AUC (mean ± std)')
axes[1].set_title('N_TYPES 평균 ± 표준편차 (5 seeds)', fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('다중 시드 N_TYPES 검증\n(N_TYPES=14가 SEED=42 artifact인지 확인)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_gnn_seed_validation.png', bbox_inches='tight', dpi=120)
plt.show()

# PSO 기준선과 비교
pso_ref = results['PSO']['PR-AUC']
print(f'\nPSO 기준선: {pso_ref:.4f}')
print('N_TYPES별 PSO 초과 빈도 (5시드 중):')
for nt in TYPES_VAL:
    beats = sum(1 for sd in SEEDS_VAL if seed_type_results[nt][sd] > pso_ref)
    mean_v = np.mean([seed_type_results[nt][sd] for sd in SEEDS_VAL])
    print(f'  N_TYPES={nt:>2}  PSO 초과: {beats}/5회  mean={mean_v:.4f}')


다중 시드 N_TYPES 검증 (Elliptic GNN)
  시드: [0, 7, 42, 77, 123]
  N_TYPES: [8, 12, 14, 17, 20]
  N_TYPES= 8  mean=0.5307 ± 0.0740  [0.38828328182353633, 0.5376440278765429, 0.6001541249331257, 0.5694355968814865, 0.5579153120941001] <-- 기본
  N_TYPES=12  mean=0.4840 ± 0.1088  [0.3967910030644099, 0.6005852896604664, 0.3683815832601476, 0.4250123066681872, 0.6293660280858631]
  N_TYPES=14  mean=0.5893 ± 0.0600  [0.6296790512216206, 0.6464897563428154, 0.6378210516421894, 0.5230000499274646, 0.509640778785979] <-- 현재최적
  N_TYPES=17  mean=0.5300 ± 0.0669  [0.572382587157992, 0.5404334927079854, 0.5757352388004136, 0.562802935551051, 0.398507867631989]
  N_TYPES=20  mean=0.5149 ± 0.0779  [0.5954230343821549, 0.5045607250520556, 0.39332024443291946, 0.6025301050406273, 0.4785134200068227]

시드별 best N_TYPES:
  SEED=  0  best=14  | {8: '0.3883', 12: '0.3968', 14: '0.6297', 17: '0.5724', 20: '0.5954'}
  SEED=  7  best=14  | {8: '0.5376', 12: '0.6006', 14: '0.6465', 17: '0.5404', 20: '0.5046'}
  SEED=

## 10. M 앙상블: 분산 감소 실험

**문제**: N_TYPES=14가 SEED=42에서 0.6378이지만 SEED=77,123에서 0.51-0.52  
**원인**: M 초기화 품질이 성능을 결정 — "M 뽑기" 가 실질적 변수  
**해법**: M 후보 K개 → warmup iter 후 에이전트 분산(diversity)이 가장 높은 M 선택 → 나머지 iter 계속

In [12]:
# ── M 앙상블: 분산 감소 실험 ──────────────────────────────────────────────────
# M 후보 K개를 warmup iter 후 diversity 기준으로 선택 → 나머지 iter 계속
M_ENSEMBLE_K = 5
WARMUP_ITER  = 20

def run_aiso_m_ensemble(X_a, n, seed, n_types=14, n_m=M_ENSEMBLE_K, warmup=WARMUP_ITER):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_a); N_a, D = Xn.shape

    best_div = -np.inf
    best_state = None

    for k in range(n_m):
        M_cand = rng2.uniform(M_LOW, W_REPEL, (n_types, n_types))
        X_cand = Xn[rng2.choice(N_a, N_AG, replace=True)].copy().astype(float)
        W_cand = rng2.dirichlet(np.ones(n_types), N_AG)
        visit_cand = np.zeros(N_a); w_r = W_REPEL

        for t in range(warmup):
            if t % 10 == 0:
                div = np.mean([np.linalg.norm(X_cand[i]-X_cand[j])
                               for i in range(N_AG) for j in range(i+1, N_AG)])
                w_r = 1.0 + 3.0 * np.exp(-div / 0.12)
            C = W_cand @ M_cand @ W_cand.T; np.fill_diagonal(C, 0)
            for i in range(N_AG):
                ci = C[i].copy(); ci[i] = 0
                ta = np.argsort(ci)[-3:]; tr2 = np.argsort(ci)[:3]
                Fv  = sum(ci[j] * (X_cand[j]-X_cand[i]) for j in ta)
                Fv += w_r * sum(ci[j] * (X_cand[j]-X_cand[i]) for j in tr2)
                Fv /= 6.0
                nn = np.argmin(np.linalg.norm(Xn - np.clip(X_cand[i]+ALPHA*Fv, 0, 1), axis=1))
                X_cand[i] = Xn[nn]; visit_cand[nn] += 1.0
                bja = ta[np.argmax(ci[ta])]
                W_cand[i] = (1-BETA)*W_cand[i] + BETA*W_cand[bja]; W_cand[i] /= W_cand[i].sum()

        # warmup 후 agent diversity로 M 품질 평가
        div_final = np.mean([np.linalg.norm(X_cand[i]-X_cand[j])
                             for i in range(N_AG) for j in range(i+1, N_AG)])
        if div_final > best_div:
            best_div = div_final
            best_state = (M_cand.copy(), X_cand.copy(), W_cand.copy(), visit_cand.copy())

    # best M으로 나머지 iter 실행
    M_use, X, W, visit = best_state
    w_r = W_REPEL
    for t in range(warmup, N_IT):
        if t % 10 == 0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1, N_AG)])
            w_r = 1.0 + 3.0 * np.exp(-div / 0.12)
        C = W @ M_use @ W.T; np.fill_diagonal(C, 0)
        for i in range(N_AG):
            ci = C[i].copy(); ci[i] = 0
            ta = np.argsort(ci)[-3:]; tr2 = np.argsort(ci)[:3]
            Fv  = sum(ci[j] * (X[j]-X[i]) for j in ta)
            Fv += w_r * sum(ci[j] * (X[j]-X[i]) for j in tr2)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn - np.clip(X[i]+ALPHA*Fv, 0, 1), axis=1))
            X[i] = Xn[nn]; visit[nn] += 1.0
            bja = ta[np.argmax(ci[ta])]
            W[i] = (1-BETA)*W[i] + BETA*W[bja]; W[i] /= W[i].sum()

    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, n, replace=True, p=probs)


# 다중 시드에서 단일 M vs 앙상블 비교
print(f'M 앙상블 분산 감소 검증 (N_TYPES=14, {len(SEEDS_VAL)}개 시드)')
print(f'  후보 M 수: {M_ENSEMBLE_K}, warmup: {WARMUP_ITER} iter')
print('='*62)

single_vals   = [seed_type_results[14][sd] for sd in SEEDS_VAL]
ensemble_vals = []

for sd in SEEDS_VAL:
    idx_ens = run_aiso_m_ensemble(X_anom_pca, N_SEEN, sd, n_types=14)
    res_ens = evaluate_gnn(idx_ens)
    ensemble_vals.append(res_ens['PR-AUC'])
    delta = res_ens['PR-AUC'] - seed_type_results[14][sd]
    print(f'  SEED={sd:>3}  단일M={seed_type_results[14][sd]:.4f}  앙상블={res_ens["PR-AUC"]:.4f}  Δ={delta:+.4f}')

pso_ref = results['PSO']['PR-AUC']
print(f'\n  단일M  mean={np.mean(single_vals):.4f} ± {np.std(single_vals):.4f}')
print(f'  앙상블 mean={np.mean(ensemble_vals):.4f} ± {np.std(ensemble_vals):.4f}')
print(f'  PSO 기준선: {pso_ref:.4f}')
print(f'  단일M  PSO 초과: {sum(1 for v in single_vals   if v > pso_ref)}/5회')
print(f'  앙상블 PSO 초과: {sum(1 for v in ensemble_vals if v > pso_ref)}/5회')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(SEEDS_VAL)); w = 0.35
axes[0].bar(x - w/2, single_vals,   w, label='단일 M',  color='#4C72B0', alpha=0.8)
axes[0].bar(x + w/2, ensemble_vals, w, label=f'M 앙상블(K={M_ENSEMBLE_K})', color='#C44E52', alpha=0.8)
axes[0].axhline(pso_ref, color='green', ls='--', lw=1.5, label=f'PSO={pso_ref:.4f}')
axes[0].set_xticks(x); axes[0].set_xticklabels([f'seed={s}' for s in SEEDS_VAL])
axes[0].set_ylabel('PR-AUC'); axes[0].set_title('단일 M vs M 앙상블 (N_TYPES=14)', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, axis='y')

axes[1].boxplot([single_vals, ensemble_vals], labels=['단일 M', f'앙상블 K={M_ENSEMBLE_K}'],
                patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].axhline(pso_ref, color='green', ls='--', lw=1.5, label=f'PSO={pso_ref:.4f}')
axes[1].set_ylabel('PR-AUC'); axes[1].set_title('분포 비교 (5 seeds)', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('M 앙상블 효과: 분산 감소 + PSO 초과 빈도 변화', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_gnn_m_ensemble.png', bbox_inches='tight', dpi=120)
plt.show()

M 앙상블 분산 감소 검증 (N_TYPES=14, 5개 시드)
  후보 M 수: 5, warmup: 20 iter
  SEED=  0  단일M=0.6297  앙상블=0.5176  Δ=-0.1121
  SEED=  7  단일M=0.6465  앙상블=0.6395  Δ=-0.0070
  SEED= 42  단일M=0.6378  앙상블=0.6077  Δ=-0.0301
  SEED= 77  단일M=0.5230  앙상블=0.4503  Δ=-0.0727
  SEED=123  단일M=0.5096  앙상블=0.5827  Δ=+0.0731

  단일M  mean=0.5893 ± 0.0600
  앙상블 mean=0.5596 ± 0.0677
  PSO 기준선: 0.6030
  단일M  PSO 초과: 3/5회
  앙상블 PSO 초과: 2/5회


## 11. 도메인 피처 기반 AISO (Yelp ALL_FEATS_ORDERED 재현)

**동기**: Yelp(`sampling_showdown.ipynb`)에서는 AISO 탐색 공간이 spam_ratio, user_conc, burst 등  
손수 설계한 도메인 피처였다. Elliptic에서는 PCA-30(원시 피처 압축)을 썼는데,  
"어떤 노드가 GCN 학습에 유용한가"를 직접 포착하는 도메인 피처로 교체하면 성능이 오르는지 검증.

| 피처 | 의미 |
|------|------|
| degree | 노드 연결 수 (그래프 중심성 proxy) |
| illicit_neigh_ratio | 직접 이웃 중 illicit 비율 |
| neigh2_illicit | 2차 이웃 illicit 접촉도 |
| ts_illicit_rate | 같은 타임스텝의 illicit 밀도 |
| local_f_{mean,std,max} | 로컬 거래 피처 통계 (f2-f94) |
| agg_f_{mean,std,max} | 집계 피처 통계 (f95-f166) |
| ts_norm | 정규화된 타임스텝 (시간 위치) |
| hub_fraud | degree × illicit_neigh_ratio (허브×사기 교차항) |

In [13]:
from sklearn.preprocessing import MinMaxScaler as MMS

# ── 도메인 피처 12차원 계산 ────────────────────────────────────────────────────
print('Elliptic 도메인 피처 계산 중...')

ei_np2 = edge_index.numpy()

# 1. degree
node_degree = np.bincount(ei_np2[0], minlength=N_NODES).astype(float)

# 2. illicit 이웃 비율
illicit_neigh_sum = np.zeros(N_NODES)
neigh_count       = np.zeros(N_NODES)
for src, dst in zip(ei_np2[0], ei_np2[1]):
    illicit_neigh_sum[src] += y_all[dst]
    neigh_count[src]       += 1
illicit_neigh_ratio = illicit_neigh_sum / np.maximum(neigh_count, 1)

# 3. 2차 이웃 illicit 접촉도
neigh2_illicit = np.zeros(N_NODES)
for src, dst in zip(ei_np2[0], ei_np2[1]):
    neigh2_illicit[src] += illicit_neigh_ratio[dst]
neigh2_illicit /= np.maximum(neigh_count, 1)

# 4. 타임스텝 illicit 밀도
ts_illicit_rate_map = {}
for ts in np.unique(ts_all):
    m = ts_all == ts
    ts_illicit_rate_map[ts] = float(y_all[m].mean()) if m.sum() > 0 else 0.0
node_ts_illicit_rate = np.array([ts_illicit_rate_map[t] for t in ts_all])

# 5-7. 로컬 피처 통계 (f2-f94 → index 1-93)
local_feat = X_scaled[:, 1:94]
local_mean = local_feat.mean(axis=1)
local_std  = local_feat.std(axis=1)
local_max  = local_feat.max(axis=1)

# 8-10. 집계 피처 통계 (f95-f166 → index 94-165)
agg_feat = X_scaled[:, 94:166]
agg_mean = agg_feat.mean(axis=1)
agg_std  = agg_feat.std(axis=1)
agg_max  = agg_feat.max(axis=1)

# 11. 정규화 타임스텝
ts_norm = (ts_all - ts_all.min()) / (ts_all.max() - ts_all.min() + 1e-8)

# 12. 허브×사기 교차항
hub_fraud = node_degree * illicit_neigh_ratio

domain_feat_raw = np.stack([
    node_degree, illicit_neigh_ratio, neigh2_illicit, node_ts_illicit_rate,
    local_mean, local_std, local_max,
    agg_mean, agg_std, agg_max,
    ts_norm, hub_fraud,
], axis=1)

domain_feat_norm = MMS().fit_transform(domain_feat_raw)
X_anom_domain    = domain_feat_norm[train_anom_idx]
print(f'도메인 피처: {domain_feat_raw.shape} → illicit pool: {X_anom_domain.shape}')

# ── PCA vs 도메인 피처 비교 실험 ─────────────────────────────────────────────
print('\n피처 공간별 AISO 비교 실행...')
dom_compare = {}

configs = [
    ('AISO(PCA-30, N=8)',  X_anom_pca,    8),
    ('AISO(PCA-30, N=14)', X_anom_pca,    14),
    ('AISO(Domain, N=12)', X_anom_domain, 12),
    ('AISO(Domain, N=14)', X_anom_domain, 14),
]

for label, X_space, nt in configs:
    print(f'  {label}...', end=' ', flush=True)
    idx = run_aiso_nt(X_space, N_SEEN, SEED, nt)
    res = evaluate_gnn(idx, label)
    dom_compare[label] = res

dom_compare['PSO(PCA-30)'] = results['PSO']

print('\n' + '='*70)
print(f'  {"방법":<28} {"피처공간":<12} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}')
print('-'*70)
for name, res in sorted(dom_compare.items(), key=lambda x: x[1]['PR-AUC'], reverse=True):
    space = 'Domain-12' if 'Domain' in name else 'PCA-30'
    star  = ' ★' if res['PR-AUC'] == max(v['PR-AUC'] for v in dom_compare.values()) else ''
    print(f'  {name:<28} {space:<12} {res["PR-AUC"]:>8.4f} {res["F1"]:>8.4f} {res["AUC"]:>8.4f}{star}')
print('='*70)

# 시각화
names_d  = list(dom_compare.keys())
aucs_d   = [dom_compare[n]['PR-AUC'] for n in names_d]
colors_d = ['#8B0000' if 'Domain' in n else '#4C72B0' if 'AISO' in n else '#55A868' for n in names_d]
order    = np.argsort(aucs_d)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar([names_d[i] for i in order], [aucs_d[i] for i in order],
              color=[colors_d[i] for i in order], alpha=0.85, edgecolor='white')
ax.axhline(results['PSO']['PR-AUC'], color='green', ls='--', lw=1.5, label=f'PSO={results["PSO"]["PR-AUC"]:.4f}')
for bar, v in zip(bars, [aucs_d[i] for i in order]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('PR-AUC'); ax.set_title('PCA-30 vs 도메인 피처 AISO 비교', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#8B0000', label='AISO (도메인 피처)'),
    Patch(facecolor='#4C72B0', label='AISO (PCA-30)'),
    Patch(facecolor='#55A868', label='PSO'),
] + [plt.Line2D([0],[0],color='green',ls='--',label=f'PSO={results["PSO"]["PR-AUC"]:.4f}')],
fontsize=9)

plt.tight_layout()
plt.savefig('elliptic_gnn_domain_feat.png', bbox_inches='tight', dpi=120)
plt.show()

Elliptic 도메인 피처 계산 중...
도메인 피처: (46564, 12) → illicit pool: (3462, 12)

피처 공간별 AISO 비교 실행...
  AISO(PCA-30, N=8)...   AISO(PCA-30, N=8)      PR-AUC=0.6002  F1=0.5775  AUC=0.8589  edges=33,438
  AISO(PCA-30, N=14)...   AISO(PCA-30, N=14)     PR-AUC=0.6378  F1=0.6068  AUC=0.8758  edges=33,446
  AISO(Domain, N=12)...   AISO(Domain, N=12)     PR-AUC=0.5640  F1=0.5525  AUC=0.8549  edges=33,466
  AISO(Domain, N=14)...   AISO(Domain, N=14)     PR-AUC=0.6153  F1=0.5899  AUC=0.8782  edges=33,474

  방법                           피처공간           PR-AUC       F1      AUC
----------------------------------------------------------------------
  AISO(PCA-30, N=14)           PCA-30         0.6378   0.6068   0.8758 ★
  AISO(Domain, N=14)           Domain-12      0.6153   0.5899   0.8782
  PSO(PCA-30)                  PCA-30         0.6030   0.6058   0.8631
  AISO(PCA-30, N=8)            PCA-30         0.6002   0.5775   0.8589
  AISO(Domain, N=12)           Domain-12      0.5640   0.5525   0.8549


In [14]:
# ── 도메인 피처 다중 시드 검증 ────────────────────────────────────────────────
# PCA(N=14): mean=0.5893 ± 0.0600, PSO 초과 3/5
# 도메인 피처(N=12,14)가 더 안정적인지 확인

TYPES_DOM = [12, 14]

dom_seed_results = {nt: {} for nt in TYPES_DOM}

print('도메인 피처 다중 시드 검증 (Elliptic GNN)')
print(f'  시드: {SEEDS_VAL}')
print(f'  N_TYPES: {TYPES_DOM}')
print('=' * 55)

for nt in TYPES_DOM:
    row = []
    for sd in SEEDS_VAL:
        idx = run_aiso_nt(X_anom_domain, N_SEEN, sd, nt)
        res = evaluate_gnn(idx)
        dom_seed_results[nt][sd] = res['PR-AUC']
        row.append(res['PR-AUC'])
    mean_v = np.mean(row); std_v = np.std(row)
    print(f'  N_TYPES={nt:>2}  mean={mean_v:.4f} ± {std_v:.4f}  {[round(v,4) for v in row]}')

print('=' * 55)

# PCA 결과와 비교
print('\n피처공간 × N_TYPES 비교 (mean ± std, PSO 초과 빈도):')
print(f'  {"방법":<28} {"mean":>7} {"std":>7}  PSO 초과')
print('-' * 55)

pso_ref = results['PSO']['PR-AUC']

# 기존 PCA 결과
for nt in [8, 14]:
    vals = [seed_type_results[nt][sd] for sd in SEEDS_VAL]
    beats = sum(1 for v in vals if v > pso_ref)
    print(f'  AISO(PCA-30,   N={nt:<2})       {np.mean(vals):>7.4f} {np.std(vals):>7.4f}  {beats}/5')

# 도메인 피처 결과
for nt in TYPES_DOM:
    vals = [dom_seed_results[nt][sd] for sd in SEEDS_VAL]
    beats = sum(1 for v in vals if v > pso_ref)
    print(f'  AISO(Domain-12, N={nt:<2})      {np.mean(vals):>7.4f} {np.std(vals):>7.4f}  {beats}/5')

print(f'\n  PSO 기준선: {pso_ref:.4f}')

# 시각화: PCA vs Domain 분포 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 시드별 곡선
ax = axes[0]
styles = {
    'PCA N=8':   ('#aaaaaa', '--'),
    'PCA N=14':  ('#4C72B0', '-'),
    'Dom N=12':  ('#DD8452', '--'),
    'Dom N=14':  ('#8B0000', '-'),
}
data_map = {
    'PCA N=8':  [seed_type_results[8][sd]         for sd in SEEDS_VAL],
    'PCA N=14': [seed_type_results[14][sd]         for sd in SEEDS_VAL],
    'Dom N=12': [dom_seed_results[12][sd]          for sd in SEEDS_VAL],
    'Dom N=14': [dom_seed_results[14][sd]          for sd in SEEDS_VAL],
}
for label, (color, ls) in styles.items():
    ax.plot(range(len(SEEDS_VAL)), data_map[label], 'o'+ls,
            color=color, ms=6, lw=2, label=label, alpha=0.85)
ax.axhline(pso_ref, color='green', ls='--', lw=1.5, label=f'PSO={pso_ref:.4f}')
ax.set_xticks(range(len(SEEDS_VAL)))
ax.set_xticklabels([f'seed={s}' for s in SEEDS_VAL])
ax.set_ylabel('PR-AUC')
ax.set_title('시드별 PR-AUC: PCA vs 도메인 피처', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 박스플롯 비교
ax = axes[1]
all_data  = [list(data_map[k]) for k in styles]
all_labels = list(styles.keys())
colors_box = ['#aaaaaa', '#4C72B0', '#DD8452', '#8B0000']
bp = ax.boxplot(all_data, labels=all_labels, patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.axhline(pso_ref, color='green', ls='--', lw=1.5, label=f'PSO={pso_ref:.4f}')
ax.set_ylabel('PR-AUC')
ax.set_title('분포 비교 (5 seeds)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

plt.suptitle('도메인 피처 vs PCA-30: 다중 시드 안정성 비교', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_gnn_domain_seed.png', bbox_inches='tight', dpi=120)
plt.show()

도메인 피처 다중 시드 검증 (Elliptic GNN)
  시드: [0, 7, 42, 77, 123]
  N_TYPES: [12, 14]
  N_TYPES=12  mean=0.6041 ± 0.0250  [0.6074, 0.6329, 0.564, 0.5902, 0.6263]
  N_TYPES=14  mean=0.5558 ± 0.0420  [0.5968, 0.5303, 0.6153, 0.511, 0.5253]

피처공간 × N_TYPES 비교 (mean ± std, PSO 초과 빈도):
  방법                              mean     std  PSO 초과
-------------------------------------------------------
  AISO(PCA-30,   N=8 )        0.5307  0.0740  0/5
  AISO(PCA-30,   N=14)        0.5893  0.0600  3/5
  AISO(Domain-12, N=12)       0.6041  0.0250  3/5
  AISO(Domain-12, N=14)       0.5558  0.0420  1/5

  PSO 기준선: 0.6030


In [15]:
# PSO 단일 시드(0.6030)만 있어 불공정 → PSO + Random도 5 시드로 확장

SEEDS_VAL = [0, 7, 42, 77, 123]

# 이미 있는 AISO 결과 재활용
multi_seed_all = {
    'AISO(PCA, N=8)':  {sd: seed_type_results[8][sd]  for sd in SEEDS_VAL},
    'AISO(PCA, N=14)': {sd: seed_type_results[14][sd] for sd in SEEDS_VAL},
    'AISO(Dom, N=12)': {sd: dom_seed_results[12][sd]  for sd in SEEDS_VAL},
    'AISO(Dom, N=14)': {sd: dom_seed_results[14][sd]  for sd in SEEDS_VAL},
}

# PSO 다중 시드
print('PSO 다중 시드 실행...')
pso_ms = {}
for sd in SEEDS_VAL:
    idx = run_pso(X_anom_pca, N_SEEN, sd)
    pso_ms[sd] = evaluate_gnn(idx)['PR-AUC']
    print(f'  SEED={sd:>3}  PSO={pso_ms[sd]:.4f}')
multi_seed_all['PSO(PCA)'] = pso_ms

# Random 다중 시드
print('\nRandom 다중 시드 실행...')
rand_ms = {}
for sd in SEEDS_VAL:
    idx = run_random(X_anom_pca, N_SEEN, sd)
    rand_ms[sd] = evaluate_gnn(idx)['PR-AUC']
    print(f'  SEED={sd:>3}  Random={rand_ms[sd]:.4f}')
multi_seed_all['Random'] = rand_ms

# ── 결과 테이블 ──────────────────────────────────────────────────────────────
pso_mean = np.mean(list(pso_ms.values()))
summary = {}
for name, sd_dict in multi_seed_all.items():
    vals = [sd_dict[sd] for sd in SEEDS_VAL]
    beats = sum(1 for v in vals if v > pso_mean)
    summary[name] = {'mean': np.mean(vals), 'std': np.std(vals),
                     'min': min(vals), 'max': max(vals), 'beats': beats, 'vals': vals}

print('\n' + '=' * 68)
print(f'  {"방법":<22} {"mean":>7} {"std":>7} {"min":>7} {"max":>7}  PSO mean 초과')
print('-' * 68)
for name, r in sorted(summary.items(), key=lambda x: x[1]['mean'], reverse=True):
    star = ' ★' if r['mean'] == max(v['mean'] for v in summary.values()) else ''
    print(f'  {name:<22} {r["mean"]:>7.4f} {r["std"]:>7.4f} {r["min"]:>7.4f} {r["max"]:>7.4f}  {r["beats"]}/5{star}')
print('=' * 68)
print(f'\n  PSO mean = {pso_mean:.4f} (5 seeds 기준선)')

# ── 시각화 ───────────────────────────────────────────────────────────────────
color_map = {
    'AISO(PCA, N=8)':  '#aaaaaa',
    'AISO(PCA, N=14)': '#4C72B0',
    'AISO(Dom, N=12)': '#C44E52',
    'AISO(Dom, N=14)': '#8B0000',
    'PSO(PCA)':        '#55A868',
    'Random':          '#cccccc',
}
ls_map = {
    'AISO(PCA, N=8)':  '--',
    'AISO(PCA, N=14)': '-',
    'AISO(Dom, N=12)': '-',
    'AISO(Dom, N=14)': '--',
    'PSO(PCA)':        '-.',
    'Random':          ':',
}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 시드별 곡선
ax = axes[0]
for name, r in summary.items():
    ax.plot(range(len(SEEDS_VAL)), r['vals'], 'o'+ls_map[name],
            color=color_map[name], ms=6, lw=2, label=name, alpha=0.9)
ax.axhline(pso_mean, color='#55A868', ls=':', lw=1.5, alpha=0.6, label=f'PSO mean={pso_mean:.4f}')
ax.set_xticks(range(len(SEEDS_VAL)))
ax.set_xticklabels([f'seed={s}' for s in SEEDS_VAL])
ax.set_ylabel('PR-AUC'); ax.set_title('시드별 PR-AUC — 전체 메서드', fontweight='bold')
ax.legend(fontsize=8, loc='lower left'); ax.grid(alpha=0.3)

# mean ± std 바 차트
ax = axes[1]
names_s = sorted(summary, key=lambda x: summary[x]['mean'], reverse=True)
bars = ax.bar(range(len(names_s)),
              [summary[n]['mean'] for n in names_s],
              yerr=[summary[n]['std'] for n in names_s],
              capsize=6, color=[color_map[n] for n in names_s],
              alpha=0.85, edgecolor='white')
ax.axhline(pso_mean, color='#55A868', ls='--', lw=1.5, label=f'PSO mean={pso_mean:.4f}')
ax.set_xticks(range(len(names_s)))
ax.set_xticklabels(names_s, rotation=25, ha='right')
ax.set_ylabel('PR-AUC (mean ± std)')
ax.set_title('5 Seeds 평균 ± 표준편차', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
for bar, n in zip(bars, names_s):
    m = summary[n]['mean']
    ax.text(bar.get_x()+bar.get_width()/2, m + summary[n]['std'] + 0.003,
            f'{m:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('전체 메서드 다중 시드 공정 비교 (PSO 포함 5 seeds)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_gnn_multiseed_final.png', bbox_inches='tight', dpi=120)
plt.show()

PSO 다중 시드 실행...
  SEED=  0  PSO=0.6670
  SEED=  7  PSO=0.5079
  SEED= 42  PSO=0.6030
  SEED= 77  PSO=0.4570
  SEED=123  PSO=0.5757

Random 다중 시드 실행...
  SEED=  0  Random=0.5080
  SEED=  7  Random=0.4985
  SEED= 42  Random=0.5538
  SEED= 77  Random=0.6555
  SEED=123  Random=0.4913

  방법                        mean     std     min     max  PSO mean 초과
--------------------------------------------------------------------
  AISO(Dom, N=12)         0.6041  0.0250  0.5640  0.6329  5/5 ★
  AISO(PCA, N=14)         0.5893  0.0600  0.5096  0.6465  3/5
  PSO(PCA)                0.5621  0.0732  0.4570  0.6670  3/5
  AISO(Dom, N=14)         0.5558  0.0420  0.5110  0.6153  2/5
  Random                  0.5414  0.0611  0.4913  0.6555  1/5
  AISO(PCA, N=8)          0.5307  0.0740  0.3883  0.6002  2/5

  PSO mean = 0.5621 (5 seeds 기준선)


In [20]:
# 원본(불균형) + Top-density 다중 시드 추가

print('원본(불균형) 다중 시드 실행...')
orig_ms = {}
for sd in SEEDS_VAL:
    idx = np.random.RandomState(sd).choice(len(train_anom_idx), N_SEEN, replace=False)
    orig_ms[sd] = evaluate_gnn(idx)['PR-AUC']
    print(f'  SEED={sd:>3}  원본={orig_ms[sd]:.4f}')
multi_seed_all['원본(불균형)'] = orig_ms

print()
print('Top-density 다중 시드 실행...')
topd_ms = {}
for sd in SEEDS_VAL:
    idx = run_topdensity(X_anom_pca, N_SEEN, sd)
    topd_ms[sd] = evaluate_gnn(idx)['PR-AUC']
    print(f'  SEED={sd:>3}  Top-density={topd_ms[sd]:.4f}')
multi_seed_all['Top-density'] = topd_ms

pso_mean = np.mean(list(multi_seed_all["PSO(PCA)"].values()))
summary2 = {}
for name, sd_dict in multi_seed_all.items():
    vals = [sd_dict[sd] for sd in SEEDS_VAL]
    beats = sum(1 for v in vals if v > pso_mean)
    summary2[name] = {"mean": np.mean(vals), "std": np.std(vals),
                      "min": min(vals), "max": max(vals), "beats": beats, "vals": vals}

print()
print('=' * 72)
print(f'  {"방법":<24} {"mean":>7} {"std":>7} {"min":>7} {"max":>7}  PSO mean 초과')
print('-' * 72)
for rank, (name, r) in enumerate(sorted(summary2.items(), key=lambda x: x[1]["mean"], reverse=True), 1):
    star = " ★" if rank == 1 else ""
    print(f"  {rank}위 {name:<22} {r["mean"]:>7.4f} {r["std"]:>7.4f} {r["min"]:>7.4f} {r["max"]:>7.4f}  {r["beats"]}/5{star}")
print('=' * 72)
print()
print(f'  PSO mean = {pso_mean:.4f} (5 seeds 기준선)')

color_map2 = {
    "AISO(Dom, N=12)": "#C44E52",
    "AISO(PCA, N=14)": "#4C72B0",
    "PSO(PCA)":        "#55A868",
    "AISO(Dom, N=14)": "#8B0000",
    "원본(불균형)":      "#888888",
    "Top-density":     "#9467bd",
    "Random":          "#cccccc",
    "AISO(PCA, N=8)":  "#dddddd",
}
ls_map2 = {
    "AISO(Dom, N=12)": "-",
    "AISO(PCA, N=14)": "-",
    "PSO(PCA)":        "-.",
    "AISO(Dom, N=14)": "--",
    "원본(불균형)":      "--",
    "Top-density":     "--",
    "Random":          ":",
    "AISO(PCA, N=8)":  ":",
}

fig, axes = plt.subplots(1, 2, figsize=(22, 6))

ax = axes[0]
for name, r in sorted(summary2.items(), key=lambda x: x[1]["mean"], reverse=True):
    ax.plot(range(len(SEEDS_VAL)), r["vals"], "o"+ls_map2.get(name, "-"),
            color=color_map2.get(name, "#aaaaaa"), ms=6, lw=2, label=name, alpha=0.9)
ax.axhline(pso_mean, color="#55A868", ls=":", lw=1.5, alpha=0.5, label=f"PSO mean={pso_mean:.4f}")
ax.set_xticks(range(len(SEEDS_VAL)))
ax.set_xticklabels([f"seed={s}" for s in SEEDS_VAL])
ax.set_ylabel("PR-AUC")
ax.set_title("시드별 PR-AUC (베이스라인 포함)", fontweight="bold")
ax.legend(fontsize=8, loc="lower left"); ax.grid(alpha=0.3)

ax = axes[1]
names_s2 = sorted(summary2, key=lambda x: summary2[x]["mean"], reverse=True)
bars = ax.bar(range(len(names_s2)),
              [summary2[n]["mean"] for n in names_s2],
              yerr=[summary2[n]["std"] for n in names_s2],
              capsize=5, color=[color_map2.get(n, "#aaaaaa") for n in names_s2],
              alpha=0.85, edgecolor="white")
ax.axhline(pso_mean, color="#55A868", ls="--", lw=1.5, label=f"PSO mean={pso_mean:.4f}")
ax.set_xticks(range(len(names_s2)))
ax.set_xticklabels(names_s2, rotation=30, ha="right")
ax.set_ylabel("PR-AUC (mean ± std)")
ax.set_title("8개 메서드 5 Seeds mean ± std", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
for bar, n in zip(bars, names_s2):
    m = summary2[n]["mean"]
    ax.text(bar.get_x()+bar.get_width()/2, m + summary2[n]["std"] + 0.003,
            f"{m:.4f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("전체 메서드 다중 시드 공정 비교 (베이스라인 포함, 5 seeds)",
             fontweight="bold", fontsize=12)
plt.tight_layout()
plt.savefig("elliptic_gnn_multiseed_final2.png", bbox_inches="tight", dpi=120)
plt.show()

원본(불균형) 다중 시드 실행...
  SEED=  0  원본=0.5545
  SEED=  7  원본=0.6190
  SEED= 42  원본=0.5752
  SEED= 77  원본=0.4685
  SEED=123  원본=0.6420

Top-density 다중 시드 실행...
  SEED=  0  Top-density=0.6293
  SEED=  7  Top-density=0.5500
  SEED= 42  Top-density=0.5701
  SEED= 77  Top-density=0.5965
  SEED=123  Top-density=0.5420

  방법                          mean     std     min     max  PSO mean 초과
------------------------------------------------------------------------
  1위 AISO(Dom, N=12)         0.6041  0.0250  0.5640  0.6329  5/5 ★
  2위 AISO(PCA, N=14)         0.5893  0.0600  0.5096  0.6465  3/5
  3위 Top-density             0.5776  0.0320  0.5420  0.6293  3/5
  4위 원본(불균형)                 0.5718  0.0602  0.4685  0.6420  3/5
  5위 PSO(PCA)                0.5621  0.0732  0.4570  0.6670  3/5
  6위 AISO(Dom, N=14)         0.5558  0.0420  0.5110  0.6153  2/5
  7위 Random                  0.5414  0.0611  0.4913  0.6555  1/5
  8위 AISO(PCA, N=8)          0.5307  0.0740  0.3883  0.6002  2/5

  PSO mean = 0.5621 (